==============================================================================
LlamaParse 잔여 실패 페이지 재처리 — 독립 실행 노트북

메인 파이프라인(build_rag_index_v5_2_pymupdf_only.ipynb)을 2단계(pdfplumber+
PyMuPDF)까지만 돌리고 100개사 본작업을 끝낸 뒤, 그중 캐스케이딩 후에도
실패가 남은 페이지만 여기서 LlamaParse로 재처리해서 기존 Chroma DB에
추가 적재함.

전제조건:
  1. 메인 파이프라인의 [Cell 7c]가 만든 extraction_diagnostics.csv가 있어야 함
     (still_failed_pages 컬럼 포함된 v5.2 이상 버전)
  2. 메인 파이프라인으로 이미 Chroma DB가 구축돼 있어야 함 (Drive 백업 경로 동일)

속도 최적화: 페이지 하나당 API 호출 1번이 아니라, 문서(doc_id) 하나당
target_pages에 실패 페이지 전체를 한꺼번에 넣어서 API 호출 1번으로 처리함
(LlamaParse는 target_pages에 콤마로 여러 페이지를 받을 수 있음).

비용 관리: SELECTION_RULE로 재처리 대상을 좁힐 수 있음 (기본: 잔여 실패
페이지가 1개라도 있으면 전부 대상 — 10개사 실측 기준 전체 비용이 작아서
굳이 더 좁힐 필요는 없었지만, 필요하면 MIN_ISSUE_RATIO를 올릴 것).

변경 이력 (Gemini 리뷰 반영):
  - [반영] Cell 3/7: add_documents 시 배치+429 재시도 로직 추가 (v6과 동일 패턴).
    DB하이텍류는 수십 페이지가 한 번에 복구돼 청크가 수백 개 나올 수 있어 필요했음
  - [반영] Cell 4: still_failed_pages 파싱을 NaN/손상값에 안전하게 방어 처리
  - [반려] "$and" 최상위 필터가 Chroma에서 지원 안 된다는 지적은 사실 아님
    (Chroma 공식 문서: 필터 객체는 필드 1개 또는 $and/$or 중 하나만 가질 수 있음
    -> 여러 조건 걸 때는 오히려 $and가 필수 문법. 기존 코드 그대로 유지)

변경 이력 (readonly DB 사고 이후 정리):
  - drive_backup_dir을 손상됐던 원래 chroma_db 대신, 마이그레이션으로 복구한
    chroma_db_migrated로 직접 지정. Drive-Drive 승격 복사는 하지 않음(그 경로
    자체가 오염 원인이었을 가능성).
  - [Cell 3] vectorstore 로드 직후 verify_writable()로 쓰기 가능 여부 즉시 확인.
    실패하면 Cell 4 이후로 진행 안 하고 바로 에러.
  - [Cell 7] 문서 단위 자체 체크포인트(llamaparse_rescue_checkpoint.json) 추가.
    문서 하나가 완전히 성공해야만 기록되므로, 중간에 세션이 끊겨도 재시작 시
    그 지점부터 정확히 이어감 (처음부터 다시 안 함).
  - [Cell 7] delete/add 각각 재시도(최대 8번, 2~15초 간격) 적용, 실패해도
    예외로 루프가 죽지 않고 다음 문서로 넘어감.
  - [Cell 7] Drive 백업 전 반드시 쓰기 검증(save_verified_backup) -> 통과해야만
    반영. 손상된 상태가 Drive를 덮어써서 재발하는 것을 원천 차단.

⚠️ 이 노트북을 실행하는 동안(특히 Cell 7)에는 컴퓨터 절전모드, 인터넷 연결 끊김,
런타임 강제 종료 등 세션이 끊기는 상황을 피할 것. 실제 쓰기(delete/add)가
일어나는 그 찰나에 끊기면 로컬 DB가 손상될 수 있음 (재시도 로직으로 대부분의
일시적 문제는 버티지만, 세션 자체가 끊기는 것은 코드로 막을 수 없음).
==============================================================================


### [Cell 1] 설치 + Drive 마운트 + API 키

In [ ]:
!pip install -q langchain langchain-google-genai langchain-chroma langchain-community \
    chromadb pandas llama-cloud-services

from google.colab import drive
drive.mount('/content/drive')

import getpass
import os

os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google AI Studio API Key 입력 (임베딩용): ")
os.environ["LLAMA_CLOUD_API_KEY"] = getpass.getpass("LlamaParse API Key 입력: ")

### [Cell 2] 설정값 — 메인 파이프라인의 CONFIG와 경로를 맞출 것

In [ ]:
CONFIG = {
    "embedding_model": "gemini-embedding-001",
    "chunk_size_tokens": 800,
    "chunk_overlap_tokens": 100,

    # 메인 파이프라인과 동일하게 맞출 것 (로컬 작업 + Drive는 백업 전용)
    "persist_dir": "/content/chroma_db_local",
    # [중요] 원래 chroma_db 경로는 손상 이력이 있어 폐기.
    # 마이그레이션으로 복구한 정상본(chroma_db_migrated)을 곧바로 정식 경로로 사용.
    # Drive-Drive 복사(승격)는 하지 않음 -- 그 과정 자체가 오염 경로였을 가능성이 있어
    # migrated 원본은 그대로 보존하고 여기서 직접 참조만 함.
    "drive_backup_dir": "/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/인덱스 결과/chroma_db_migrated",

    "extraction_diag_csv": "/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/인덱스 결과/extraction_diagnostics.csv",

    # LlamaParse
    "llamaparse_mode": "cost_effective",  # 페이지당 3크레딧(≈$0.00375). 필요시 "fast"(1크레딧)
    "llamaparse_language": "ko",

    # 재처리 대상 선정 규칙 — 기본은 "잔여 실패 페이지가 1개라도 있으면 전부".
    # 예산이 걱정되면 0.3 등으로 올려서 심한 문서만 우선 처리 가능.
    "min_issue_ratio_after": 0.0,

    # [신규] Chroma add_documents 시 배치 크기/딜레이 — DB하이텍처럼 수십 페이지가
    # 한꺼번에 복구되면 청크가 수백 개가 나올 수 있어, 한 번에 넣으면 429 위험 있음
    "embed_batch_size": 100,
    "embed_batch_delay_sec": 1.0,

    # [신규] 몇 건 처리마다 중간 백업할지 — 루프 도중 멈춰도(429, 실수로 중지 등)
    # 최근 백업 시점까지만 다시 하면 되도록 손실 범위를 제한함
    "backup_every_n_docs": 3,
}

### [Cell 3] 라이브러리 import + 로컬 Chroma 복원

In [ ]:
import ast
import json
import time
import shutil

import pandas as pd
import tiktoken
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

try:
    _enc = tiktoken.encoding_for_model(CONFIG["embedding_model"])
except KeyError:
    _enc = tiktoken.get_encoding("cl100k_base")


def token_len(text: str) -> int:
    return len(_enc.encode(text))


splitter = RecursiveCharacterTextSplitter(
    chunk_size=CONFIG["chunk_size_tokens"],
    chunk_overlap=CONFIG["chunk_overlap_tokens"],
    length_function=token_len,
    separators=["\n\n", "\n", ". ", " ", ""],
)


# 메인 파이프라인과 동일한 복원 로직 (Drive에 직접 Chroma를 물리면 손상 위험 있음 — 실제로 겪은 사고)
def restore_local_from_drive_backup():
    if os.path.exists(CONFIG["persist_dir"]):
        print("로컬 인덱스 이미 존재 (복원 스킵)")
        return
    if os.path.exists(CONFIG["drive_backup_dir"]):
        print("Drive 백업 -> 로컬 복원 중")
        shutil.copytree(CONFIG["drive_backup_dir"], CONFIG["persist_dir"])
        print("복원 완료")
    else:
        raise FileNotFoundError(
            "Drive 백업이 없음 — 메인 파이프라인을 먼저 돌려서 Chroma DB를 만들어야 함"
        )


def backup_local_to_drive():
    shutil.copytree(CONFIG["persist_dir"], CONFIG["drive_backup_dir"], dirs_exist_ok=True)


# [신규] v6 파이프라인과 동일한 배치+재시도 로직 재사용 (429 방지)
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_message


@retry(
    wait=wait_exponential(multiplier=2, min=5, max=90),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_message(match=".*RESOURCE_EXHAUSTED.*|.*429.*"),
    reraise=True,
)
def _add_batch_with_retry(vectorstore, batch_chunks):
    ids = [c.metadata["chunk_id"] for c in batch_chunks]
    vectorstore.add_documents(batch_chunks, ids=ids)


def add_documents_throttled(vectorstore, chunks, batch_size, delay_sec):
    """청크를 배치로 나눠서 넣고, 배치 사이에 대기 -> 429(RESOURCE_EXHAUSTED) 완화."""
    total = len(chunks)
    for i in range(0, total, batch_size):
        batch = chunks[i:i + batch_size]
        _add_batch_with_retry(vectorstore, batch)
        if i + batch_size < total:
            time.sleep(delay_sec)


restore_local_from_drive_backup()
embeddings = GoogleGenerativeAIEmbeddings(model=CONFIG["embedding_model"])
vectorstore = Chroma(persist_directory=CONFIG["persist_dir"], embedding_function=embeddings)
print("기존 Chroma DB 로드 완료")

# [신규] 로드 직후 실제 쓰기 가능 여부 검증 -- 여기서 걸리면 Cell 4~6 낭비 없이
# 바로 알 수 있음. readonly 손상 DB는 이 시점에 확실히 걸러짐.
def verify_writable(vs):
    """실제 쓰기(delete+재적재)가 되는지 검증. [수정] 재적재 시 embeddings를
    누락하면 Chroma 저수준 API가 기본 임베딩 함수(384차원)로 자동 재계산을
    시도해서, 원래 컬렉션(gemini-embedding-001, 3072차원)과 차원이 안 맞아
    에러가 남 -- DB 문제가 아니라 검증 코드 자체의 버그였음. embeddings까지
    같이 가져와서 그대로 넘기도록 수정."""
    t = vs.get(limit=1, include=[])
    if not t["ids"]:
        print("컬렉션이 비어있어 쓰기 테스트 스킵 (읽기는 됐으므로 일단 진행)")
        return True
    try:
        target_id = t["ids"][0]
        full = vs.get(ids=[target_id], include=["documents", "metadatas", "embeddings"])
        vs.delete(ids=[target_id])
        vs._collection.add(
            ids=[target_id],
            documents=full["documents"],
            metadatas=full["metadatas"],
            embeddings=full["embeddings"],  # [수정] 원래 임베딩 그대로 재사용, 재계산 안 함
        )
        return True
    except Exception as e:
        print(f"❌ 쓰기 검증 실패: {e}")
        return False


if verify_writable(vectorstore):
    print(f"✅ 쓰기 검증 통과 -- 총 청크: {vectorstore._collection.count()}")
else:
    raise RuntimeError(
        "이 DB는 지금 쓰기 불가 상태입니다. Cell 4 이후로 진행하지 마세요. "
        "drive_backup_dir 경로 자체가 손상됐을 수 있으니 원인부터 확인할 것."
    )

### [Cell 4] 재처리 대상 선정 — extraction_diagnostics.csv에서 잔여 실패 문서만 추출

In [ ]:
diag_df = pd.read_csv(CONFIG["extraction_diag_csv"])

targets = diag_df[
    (diag_df["issue_pages_after"] > 0)
    & (diag_df["issue_ratio_after"] >= CONFIG["min_issue_ratio_after"])
].copy()

# still_failed_pages는 CSV에 문자열(JSON 리스트)로 저장돼 있으므로 파싱.
# [신규] NaN이나 빈 값이 섞여 있어도 json.loads에서 죽지 않게 방어적으로 처리
# (targets은 issue_pages_after > 0으로 이미 필터링돼 있어 정상 케이스면 항상
# 값이 있어야 하지만, CSV가 엑셀 등을 거치며 손상될 가능성을 감안한 안전장치)
def safe_parse_pages(val):
    if pd.isna(val) or not str(val).strip():
        return []
    try:
        return json.loads(val)
    except (json.JSONDecodeError, TypeError):
        return []


targets["still_failed_pages"] = targets["still_failed_pages"].apply(safe_parse_pages)

_empty_after_parse = targets[targets["still_failed_pages"].apply(len) == 0]
if not _empty_after_parse.empty:
    print(f"⚠️ 파싱 후 빈 리스트가 된 문서 {len(_empty_after_parse)}건 — CSV 손상 의심, "
          f"확인 필요: {_empty_after_parse['doc_id'].tolist()}")

total_pages = targets["still_failed_pages"].apply(len).sum()
print(f"재처리 대상: {len(targets)}개 문서, 총 {total_pages}페이지")
print(targets[["company_code", "doc_type", "issue_pages_after", "issue_ratio_after"]].to_string(index=False))

# 실제 API 호출 전 예상 크레딧/비용 확인
_CREDITS_PER_PAGE = 3 if CONFIG["llamaparse_mode"] == "cost_effective" else 1
est_credits = total_pages * _CREDITS_PER_PAGE
print(f"\n예상 크레딧: {est_credits} (약 ${est_credits * 0.00125:.2f}) "
      f"— 무료 월 10,000크레딧 안에 들어오는지 확인할 것")

### [Cell 5] LlamaParse 재처리 함수 — 문서당 API 호출 1번 (모든 실패 페이지 한꺼번에)

In [ ]:
_LLAMAPARSE_MODE_MAP = {
    "fast": "parse_page_without_llm",
    "cost_effective": "parse_page_with_llm",
}


def llamaparse_rescue_pages(file_path: str, page_numbers_1indexed: list[int]) -> dict[int, str]:
    """실패 페이지 전체를 한 번의 API 호출로 재처리.
    반환: {1-indexed 페이지 번호: 마크다운 텍스트}

    주의: LlamaParse의 target_pages는 0-indexed이므로 변환 필요.
    결과 순서는 요청한 페이지 오름차순으로 반환된다고 가정 — 처음 실행 시
    페이지 번호가 실제로 맞는지 반드시 원본 PDF와 대조해서 확인할 것.
    """
    from llama_cloud_services import LlamaParse  # noqa: 지연 import

    sorted_pages = sorted(page_numbers_1indexed)
    zero_indexed = [p - 1 for p in sorted_pages]
    target_pages_str = ",".join(str(p) for p in zero_indexed)

    parser = LlamaParse(
        api_key=os.environ.get("LLAMA_CLOUD_API_KEY"),
        result_type="markdown",
        parse_mode=_LLAMAPARSE_MODE_MAP.get(CONFIG["llamaparse_mode"], "parse_page_with_llm"),
        language=CONFIG["llamaparse_language"],
        target_pages=target_pages_str,
    )
    result = parser.parse(file_path)
    docs = result.get_markdown_documents(split_by_page=True)

    if len(docs) != len(sorted_pages):
        # [260904 QA 수정] 예전에는 경고만 찍고 위치 기반 zip을 그대로 진행해서,
        # LlamaParse가 페이지를 누락/재배열해 반환하면 실제보다 밀린 페이지 번호가
        # 그대로 청크 메타데이터에 저장됐다 (P-1-1/P-1-2 등 "존재하지 않는 페이지"
        # 인용 오류의 원인 중 하나). 지금은 개수가 다르면 이 문서를 통째로 건너뛴다 —
        # 잘못된 페이지가 섞인 채 인덱싱되는 것보다, 사람이 재확인하는 편이 안전하다.
        # 호출부(Cell 7)가 이미 Exception을 잡아서 건너뛰고 로그에 남기므로 안전하게
        # 이 문서만 스킵되고 다음 실행 때 재시도된다.
        raise RuntimeError(
            f"요청 페이지 수({len(sorted_pages)})와 반환 문서 수({len(docs)})가 다름 — "
            f"LlamaParse가 일부 페이지를 누락/재배열했을 가능성이 커서 위치 기반 매칭을 "
            f"중단합니다. 이 문서({file_path})는 건너뛰고 별도로 재확인할 것 "
            f"(요청 페이지: {sorted_pages})."
        )

    return {page: doc.text for page, doc in zip(sorted_pages, docs)}

### [Cell 6] (권장) 1개 문서로 먼저 소규모 테스트
본격 루프 전에, 페이지 매칭이 실제로 맞는지 원본 PDF와 대조해서 확인할 것.

In [ ]:
_test_row = targets.iloc[0]
_test_pages = _test_row["still_failed_pages"][:2]  # 처음 2페이지만
print(f"테스트: {_test_row['doc_id']}, 페이지 {_test_pages}")

_test_result = llamaparse_rescue_pages(_test_row["file_path"], _test_pages)
for page, text in _test_result.items():
    print(f"\n--- p.{page} (앞 300자) ---")
    print(text[:300])

print("\n원본 PDF를 열어서 위 페이지 번호·내용이 실제로 일치하는지 확인한 뒤 다음 셀로 진행할 것.")

### [Cell 7] 본 실행 — 전체 재처리 + Chroma에 추가 적재

In [ ]:
import time
import json

# ============================================================
# 문서 단위 진행 상황을 별도로 추적 -- 이미 완료된 문서는 재시작해도
# 자동으로 건너뜀 (재개 시 처음부터 다시 안 함). Chroma의 checkpoint랑
# 별개로, 이 노트북 전용 진행 기록.
# ============================================================
RESCUE_CHECKPOINT_PATH = CONFIG["extraction_diag_csv"].replace(
    "extraction_diagnostics.csv", "llamaparse_rescue_checkpoint.json"
)

if os.path.exists(RESCUE_CHECKPOINT_PATH):
    with open(RESCUE_CHECKPOINT_PATH, encoding="utf-8") as f:
        rescue_checkpoint = json.load(f)
else:
    rescue_checkpoint = {}

print(f"이미 완료된 문서: {len(rescue_checkpoint)}건 -- 이건 건너뜀")

BACKUP_EVERY_N_DOCS = 5  # 몇 문서마다 Drive 백업할지 (checkpoint는 매 문서 기록, 백업만 이 주기)


def _retry(fn, label, max_attempts=8, base_wait=2, max_wait=15):
    """재시도: 2,4,6,8,10,12,14,15초 정도로 촘촘하게, 최대 8번.
    실패해도 예외를 던지지 않고 False를 반환해서 루프가 안 죽음."""
    for attempt in range(max_attempts):
        try:
            fn()
            return True
        except Exception as e:
            wait = min(base_wait * (attempt + 1), max_wait)
            print(f"    ⚠️ {label} 실패 ({attempt+1}/{max_attempts}): {e} -- {wait}초 대기")
            time.sleep(wait)
    print(f"    ❌ {label}: {max_attempts}번 재시도해도 실패 -- 건너뜀")
    return False


def save_verified_backup():
    """지금 상태가 실제로 쓰기 가능한지 확인하고 나서만 Drive에 반영.
    손상된 상태가 Drive를 덮어써서 재발하는 걸 막기 위함 (과거 사고의 원인)."""
    if not verify_writable(vectorstore):
        print("  ⚠️ 현재 상태 검증 실패 -- 백업 건너뜀 (이전 정상 백업 유지됨)")
        return False
    backup_local_to_drive()
    return True


# 본 실행 전 1회 재검증 (Cell 3에서 이미 했지만, 다른 셀 실행 사이 시간이 지났을 수 있어 재확인)
if not verify_writable(vectorstore):
    raise RuntimeError("❌ 쓰기 불가 상태 -- 본 실행을 시작하지 않습니다.")
print("✅ 쓰기 가능 재확인 -- 본 실행 시작\n")


# ============================================================
# 본 실행 -- 문서 하나 = 원자적 단위. 완전히 성공해야만 checkpoint에 기록
# ============================================================
rescue_log = []

for _, row in targets.iterrows():
    doc_id = row["doc_id"]

    if doc_id in rescue_checkpoint:
        continue  # 이미 완료됨, 조용히 스킵

    file_path = row["file_path"]
    pages = row["still_failed_pages"]
    print(f"처리 중: {doc_id} ({len(pages)}페이지)")

    try:
        rescued = llamaparse_rescue_pages(file_path, pages)
    except Exception as e:
        print(f"  ❌ LlamaParse API 실패 (건너뜀, 다음에 재시도됨): {e}")
        rescue_log.append({"doc_id": doc_id, "status": "api_error", "detail": str(e)})
        continue

    doc_success = True

    # 기존 청크 삭제 후 교체 (중복 적재 방지)
    for page in pages:
        def _do_delete(page=page):
            existing = vectorstore.get(where={"$and": [{"doc_id": doc_id}, {"page": page}]})
            if existing["ids"]:
                vectorstore.delete(ids=existing["ids"])

        if not _retry(_do_delete, f"p.{page} 삭제"):
            doc_success = False

    new_chunks = []
    for page, text in rescued.items():
        if not text.strip():
            continue
        for i, chunk_text in enumerate(splitter.split_text(text)):
            if len(chunk_text.strip()) < 20:
                continue
            # 메인 파이프라인의 순차 counter와 절대 겹치지 않도록 "llama" 프리픽스로 별도 네임스페이스
            chunk_id = f"{doc_id}_llama_p{page:04d}_{i:02d}"
            meta = {
                "company_code": row["company_code"], "company_name": row["company_name"],
                "year": int(row["year"]), "doc_type": row["doc_type"],
                "doc_id": doc_id, "chunk_id": chunk_id, "page": page,
                "section_title": "", "content_type": "text", "table_data": "",
                "esg_area": "", "applicable_item_codes": "[]",
                "embedding_model": CONFIG["embedding_model"],
                "indexed_at": pd.Timestamp.now().isoformat(),
                "source_file_hash": "llamaparse_rescue", "extraction_source": "llamaparse",
                "chunk_char_count": len(chunk_text),
            }
            new_chunks.append(Document(page_content=chunk_text, metadata=meta))

    if new_chunks:
        def _do_add():
            add_documents_throttled(
                vectorstore, new_chunks,
                batch_size=CONFIG["embed_batch_size"],
                delay_sec=CONFIG["embed_batch_delay_sec"],
            )
        if _retry(_do_add, "청크 적재"):
            print(f"  -> {len(new_chunks)}개 청크 적재 완료")
        else:
            doc_success = False

    # [핵심] 이 문서가 완전히 성공했을 때만 checkpoint 기록
    # -> 여기서 멈추면, 재시작 시 이 문서부터 다시 하면 되고 이전 문서들은 안전함
    if doc_success:
        rescue_checkpoint[doc_id] = True
        with open(RESCUE_CHECKPOINT_PATH, "w", encoding="utf-8") as f:
            json.dump(rescue_checkpoint, f, ensure_ascii=False, indent=2)
        rescue_log.append({
            "doc_id": doc_id, "status": "ok",
            "pages_requested": len(pages), "chunks_added": len(new_chunks),
        })

        n_completed = sum(1 for r in rescue_log if r["status"] == "ok")
        if n_completed % BACKUP_EVERY_N_DOCS == 0:
            print(f"  --- {n_completed}건 완료, 검증 후 백업 시도 ---")
            if save_verified_backup():
                print(f"  💾 백업 완료")
    else:
        print(f"  ⚠️ {doc_id} 일부 실패 -- checkpoint 기록 안 함 (다음 실행 시 재시도됨)")
        rescue_log.append({"doc_id": doc_id, "status": "partial_fail"})

    print()

print(f"\n완료: {len(rescue_log)}건 처리")
print("최종 검증 + 백업 중...")
save_verified_backup()
print("종료")


### [Cell 8] 결과 로그 저장 + 요약

In [ ]:
log_df = pd.DataFrame(rescue_log)
log_path = CONFIG["extraction_diag_csv"].replace(
    "extraction_diagnostics.csv", "llamaparse_rescue_log.csv"
)
log_df.to_csv(log_path, index=False, encoding="utf-8-sig")
print(f"로그 저장: {log_path}")
print(log_df.to_string(index=False))

if "status" in log_df.columns:
    n_ok = (log_df["status"] == "ok").sum()
    n_err = (log_df["status"] == "api_error").sum()
    print(f"\n성공: {n_ok}건 / 실패: {n_err}건")
    if n_err > 0:
        print("실패 건은 이 노트북을 다시 실행하면 됨 — Chroma add는 chunk_id 기반이라 "
              "성공한 문서를 다시 돌려도 중복 적재되지 않음 (동일 ID면 덮어씀).")